# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id` fields as defined in the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running in a fresh environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect high-level properties of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Version: {getattr(metadata,'version',None)}\nLicense: {getattr(metadata,'license',None)}\nPublished: {getattr(metadata,'datePublished',None)}")

## 2. Data Overview
Review available record sets, their `@id`s, and associated fields/columns. All Croissant entities are referenced by their `@id` property.

In [ ]:
# List all record sets in the dataset by their @id and name (if available)
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for recset in record_sets:
    print(f"  - @id: {recset.id}, name: {getattr(recset,'name',None)}")

# For each record set, print its fields and columns by @id
for recset in record_sets:
    print(f"\nRecord Set @id: {recset.id}")
    print("  Fields/Columns (by @id):")
    # Croissant's fields and columns
    # Prefer 'fields' if available, fallback to 'columns'
    fields = getattr(recset,'fields',None)
    if fields is None:
        # Some schemas use 'columns'
        fields = getattr(recset,'columns',None)
    if fields:
        for f in fields:
            print(f"    - @id: {f.id}, name: {getattr(f,'name',None)}, dataType: {getattr(f,'data_type',None)}")
    else:
        print("    (No fields/columns found in schema)")

## 3. Data Extraction
Demonstrate how to extract data from a record set using its `@id`. The main record set contains the protein measurements table.

In [ ]:
# Use the main record set's @id
# Use the first one for demonstration (adjust as necessary)
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id
print(f"Extracting from record set @id: {main_record_set_id}")

# Load all records from this record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns in main DataFrame:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and aggregate quantities from the main record set using the fields referenced by their `@id`.

In [ ]:
# Identify a numeric field in the main table
# After running the previous cell, review the column list.
# For demonstration, let's use a likely field name -- please check your dataset for exact column names, or re-run the previous cell to inspect column names.
numeric_field_id = None
# Try some common protein abundance/intensity fields
for candidate in ['normalized_abundance', 'intensity', 'MW', 'molecular_weight', 'peptide_count', 'number_of_peptides']:
    if candidate in df.columns:
        numeric_field_id = candidate
        break
if numeric_field_id is None:
    # Default to the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    raise RuntimeError("Cannot find a numeric field for demonstration. Please check your dataset.")
print(f"Using numeric field: '{numeric_field_id}' (@id)")

# Define a threshold for filtering -- we'll use the median as a demo
threshold = df[numeric_field_id].median()

# Filter data
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {filtered_df.shape[0]} out of {df.shape[0]}")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (z-score):")
print(filtered_df[[numeric_field_id, normalized_col]].head())

# Group by a key field, e.g., 'accession' or 'description', if present
group_field_id = None
for candidate in ['description', 'accession', 'protein_id', 'gene_name']:
    if candidate in df.columns:
        group_field_id = candidate
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field, and optionally, relationships to a grouping field. All chart axes are labeled with the respective field `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(f"{numeric_field_id} (@id)")
plt.title(f"Distribution of {numeric_field_id} in main record set")
plt.show()

# If grouping field is present, plot mean value by group (first 20 groups for readability)
if group_field_id:
    plt.figure(figsize=(10,4))
    ordering = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(data=ordering, x=group_field_id, y=numeric_field_id)
    plt.title(f"Top 20 mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(f"{group_field_id} (@id)")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded the dataset and accessed its metadata and record set structure via their `@id`s.
- Extracted and inspected the main data table (protein records).
- Performed basic data processing and normalization using fields referenced by their Croissant `@id`.
- Visualized the distribution of protein abundance and summarized the data by grouped attributes if available.

This workflow demonstrates FAIR principles (Findable, Accessible, Interoperable, Reusable) applied to real scientific data with the Croissant and mlcroissant ecosystem. You can extend this notebook for further domain analysis or machine learning workflows, always referencing original schema `@id`s for unambiguous documentation.